# Clean Raw Weather Data

Notebook làm sạch dữ liệu raw từ `data/raw/Hanoi.csv` và lưu ra `data/processed/Hanoi_clean.csv`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data/raw/Hanoi.csv").exists() and (PROJECT_ROOT.parent / "data/raw/Hanoi.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "data/raw/Hanoi.csv"
OUTPUT_PATH = PROJECT_ROOT / "data/processed/Hanoi_clean.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(RAW_PATH)
print(OUTPUT_PATH)

## 1. Đọc dữ liệu raw

In [ ]:
df = pd.read_csv(RAW_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
df.head()

## 2. Chuẩn hoá thời gian và xoá trùng

In [ ]:
df["time"] = pd.to_datetime(df["time"], errors="coerce")

invalid_time = df["time"].isna().sum()
duplicate_time = df["time"].duplicated().sum()
duplicate_rows = df.duplicated().sum()

df = df.dropna(subset=["time"])
df = df.sort_values("time")
df = df.drop_duplicates(subset=["time"], keep="first")
df = df.reset_index(drop=True)

print(f"Invalid timestamps before cleaning: {invalid_time:,}")
print(f"Duplicate timestamps before cleaning: {duplicate_time:,}")
print(f"Duplicate rows before cleaning: {duplicate_rows:,}")
print(f"Time range: {df['time'].min()} -> {df['time'].max()}")

## 3. Chuẩn hoá kiểu dữ liệu

In [ ]:
for column in df.columns:
    if column in ["time", "city"]:
        continue

    converted = pd.to_numeric(df[column], errors="coerce")
    original_non_null = df[column].notna().sum()
    converted_non_null = converted.notna().sum()

    if original_non_null == 0 or converted_non_null / original_non_null >= 0.9:
        df[column] = converted

df.dtypes

## 4. Xử lý missing values

In [ ]:
missing_before = df.isna().sum().sort_values(ascending=False)
missing_before[missing_before > 0].head(30)

In [ ]:
numeric_columns = df.select_dtypes(include=[np.number]).columns
object_columns = df.select_dtypes(exclude=[np.number, "datetime64[ns]"]).columns

df[numeric_columns] = df[numeric_columns].interpolate(method="linear", limit_direction="both")
for column in numeric_columns:
    if df[column].isna().any():
        df[column] = df[column].fillna(df[column].median())

for column in object_columns:
    df[column] = df[column].ffill().bfill()

missing_after = df.isna().sum().sum()
print(f"Total missing after cleaning: {missing_after:,}")

## 5. Kiểm tra khoảng thời gian

In [ ]:
time_diff = df["time"].diff().dropna()
print(f"Most common interval: {time_diff.mode().iloc[0]}")
print(f"Non-hourly intervals: {(time_diff != pd.Timedelta(hours=1)).sum():,}")

## 6. Lưu dữ liệu sạch

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")